# Can Math Play Taboo?

**Thursday Masking Day · Tilburg Multiscale Summerschool 2026 · Take-home notebook**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WimPouw/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/07_taboo_geometry.ipynb)

---

The BalanceCorpus game has a simple structure: describe a target word without using  
five forbidden (taboo) words. For example:

> **Target:** doughnut &nbsp;|&nbsp; **Taboo:** hole, bagel, round, food, pastry

Humans navigate this intuitively. But what would a computer do?

It turns out that words have *geometry*. Words with similar meanings sit close together  
in a high-dimensional space learned from millions of texts. We can use that geometry to:
- find the words nearest to a target
- exclude the ones blocked by the taboo set
- see which targets are hardest to describe

### What you will do

| Step | Task |
|------|------|
| 1 | Load the BalanceCorpus target + taboo words |
| 2 | Visualise the semantic neighbourhood of one target |
| 3 | Find the computer's best clue words (near target, far from taboos) |
| 4 | Map all 35 targets in 2D semantic space |
| 5 | Rank targets by how "hard" their taboo sets make them |

No prior NLP knowledge required.

## Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q spacy matplotlib pandas scikit-learn plotly
!{sys.executable} -m spacy download en_core_web_md -q

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

plt.rcParams.update({
    'figure.facecolor': '#161616', 'axes.facecolor': '#262626',
    'axes.edgecolor': '#525252', 'axes.labelcolor': '#c6c6c6',
    'xtick.color': '#6f6f6f', 'ytick.color': '#6f6f6f',
    'text.color': '#f4f4f4', 'grid.color': '#393939', 'grid.linestyle': '--',
})

PLOTLY_DARK = dict(
    paper_bgcolor='#161616', plot_bgcolor='#262626',
    font_color='#f4f4f4',
    xaxis=dict(gridcolor='#393939', zerolinecolor='#525252'),
    yaxis=dict(gridcolor='#393939', zerolinecolor='#525252'),
    legend=dict(bgcolor='rgba(38,38,38,0.8)'),
)

nlp = spacy.load('en_core_web_md')
print('Ready.')

## 1 — Load the corpus metadata

In [ ]:
META_URL = ('https://raw.githubusercontent.com/WimPouw/'
            'TilburgMultiscaleSummerschool2026/main/'
            'Datasets/BalanceCorpus/metadata.csv')

try:
    meta = pd.read_csv(META_URL)
    print(f'Loaded from GitHub: {len(meta)} trials')
except Exception:
    # Fallback: a small hand-typed sample if offline
    meta = pd.DataFrame({
        'target_word': ['doughnut','spinach','balloon','bacon','tiger',
                        'traffic','cologne','hunger','beauty','music'],
        'taboo_1': ['hole','vegetable','helium','pork','feline',
                    'cars','smell','starving','pretty','song'],
        'taboo_2': ['bagel','eat','dirigible','pig','lion',
                    'road','perfume','thirsty','ugly','sound'],
        'taboo_3': ['round','green','round','egg','stripe',
                    'jam','spray','food','attractive','rhythm'],
        'taboo_4': ['food','leafy','air','side','stripes',
                    'congestion','scent','diet','gorgeous','melody'],
        'taboo_5': ['pastry','popeye','blow','ham','zoo',
                    'highway','cologne','eating','handsome','beat'],
        'clue_giver_condition': ['board','board','ground','ground','board',
                                  'ground','board','ground','board','ground'],
    })
    print('Using offline fallback sample.')

TABOO_COLS = [c for c in meta.columns if c.startswith('taboo_')]

# One row per unique target word
targets = (
    meta[['target_word'] + TABOO_COLS]
    .drop_duplicates('target_word')
    .reset_index(drop=True)
)
print(f'Unique targets: {len(targets)}')
targets.head()

## 2 — Word vectors: what are they?

A word vector is a list of ~300 numbers that places a word in space.  
Words with similar meanings have similar vectors — you can measure the similarity  
with **cosine similarity**: 1.0 = identical direction, 0.0 = unrelated.

Let's check a few pairs:

In [ ]:
def sim(w1: str, w2: str) -> float:
    return nlp(w1).similarity(nlp(w2))

pairs = [
    ('doughnut', 'bagel'),
    ('doughnut', 'tiger'),
    ('tiger', 'lion'),
    ('tiger', 'spinach'),
    ('music', 'melody'),
    ('hunger', 'starvation'),
]

print(f'{"Word 1":<12} {"Word 2":<12}  similarity')
print('-' * 38)
for w1, w2 in pairs:
    print(f'{w1:<12} {w2:<12}  {sim(w1, w2):.3f}')

## 3 — Visualise one target and its taboo zone

In [ ]:
TARGET = 'doughnut'    # change this to any target in the corpus

row = targets[targets['target_word'] == TARGET].iloc[0]
taboos = [row[c] for c in TABOO_COLS if pd.notna(row[c])]

CANDIDATE_POOL = [
    'sweet', 'fried', 'glaze', 'sugar', 'cake', 'ring', 'sprinkle',
    'bread', 'breakfast', 'coffee', 'pastry', 'circle', 'snack',
    'bake', 'frosting', 'dessert', 'torus', 'cruller',
    'muffin', 'cookie', 'pretzel', 'waffle', 'eclair',
]

all_words = [TARGET] + taboos + CANDIDATE_POOL
vecs      = np.array([nlp(w).vector for w in all_words])
n_taboo   = len(taboos)

coords = PCA(n_components=2).fit_transform(vecs)

roles = ['target'] + ['taboo'] * n_taboo + ['candidate'] * len(CANDIDATE_POOL)
df_plot = pd.DataFrame({
    'word': all_words,
    'x':    coords[:, 0],
    'y':    coords[:, 1],
    'role': roles,
})

COLOR_MAP  = {'target': '#f1c21b', 'taboo': '#da1e28', 'candidate': '#78a9ff'}
SYMBOL_MAP = {'target': 'star',    'taboo': 'x',       'candidate': 'circle'}
SIZE_MAP   = {'target': 18,        'taboo': 12,         'candidate': 8}

df_plot['size'] = df_plot['role'].map(SIZE_MAP)

fig = px.scatter(
    df_plot, x='x', y='y',
    color='role', color_discrete_map=COLOR_MAP,
    symbol='role', symbol_map=SYMBOL_MAP,
    size='size', size_max=18,
    text='word',
    hover_data={'word': True, 'role': True, 'x': ':.3f', 'y': ':.3f', 'size': False},
    title=f'Semantic neighbourhood of "{TARGET}" — hover to explore',
    labels={'x': 'PC 1', 'y': 'PC 2', 'role': ''},
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(**PLOTLY_DARK, height=520)
fig.show()

## 4 — Find the computer's best clue words

**Strategy:** find words that are *near the target* AND *far from all taboo words*.

We score each candidate as:
```
score = similarity(candidate, target) − max(similarity(candidate, taboo_i))
```
A high score means: close to the target, but doesn't tread on any taboo word.

In [ ]:
def best_clues(target, taboos, candidates, top_n=8):
    t_vec   = nlp(target)
    tb_vecs = [nlp(w) for w in taboos]

    rows = []
    for cand in candidates:
        c_vec      = nlp(cand)
        sim_target = t_vec.similarity(c_vec)
        sim_taboos = [tb.similarity(c_vec) for tb in tb_vecs]
        score      = sim_target - max(sim_taboos)
        rows.append({
            'word':         cand,
            'sim_target':   round(sim_target, 3),
            'max_sim_taboo': round(max(sim_taboos), 3),
            'score':        round(score, 3),
        })

    df = pd.DataFrame(rows).sort_values('score', ascending=False)
    return df.head(top_n).reset_index(drop=True)


clues = best_clues(TARGET, taboos, CANDIDATE_POOL)
print(f'Best clue words for "{TARGET}"  (forbidden: {taboos}):')
print(clues.to_string(index=False))

## 5 — All 35 targets in one semantic map

In [ ]:
target_words = targets['target_word'].tolist()
t_vecs  = np.array([nlp(w).vector for w in target_words])
coords2 = PCA(n_components=2).fit_transform(t_vecs)

df_all = pd.DataFrame({
    'word': target_words,
    'x':    coords2[:, 0],
    'y':    coords2[:, 1],
})

fig = px.scatter(
    df_all, x='x', y='y', text='word',
    hover_data={'word': True, 'x': ':.3f', 'y': ':.3f'},
    title='All target words in 2D semantic space — hover for word, drag to zoom',
    labels={'x': 'PC 1', 'y': 'PC 2'},
)
fig.update_traces(
    textposition='top center', textfont_size=8,
    marker=dict(color='#78a9ff', size=9),
)
fig.update_layout(**PLOTLY_DARK, height=560)
fig.show()

print("Words that cluster together tend to be easier to confuse with each other.")
print("Try to spot groups — animals, food, abstract concepts.")

## 6 — Which targets are hardest to describe?

A target is hard if its taboo words cover a large portion of its semantic neighbourhood —  
i.e. the taboos are *close to the target* AND *close to the nearest candidate clue words*.

We estimate difficulty as the **mean similarity between the target and its taboo words**.
High similarity → the taboos block the most obvious paths.

In [ ]:
difficulties = []
for _, row in targets.iterrows():
    tword       = row['target_word']
    taboo_words = [row[c] for c in TABOO_COLS if pd.notna(row[c])]
    t_vec       = nlp(tword)
    sims        = [t_vec.similarity(nlp(tb)) for tb in taboo_words]
    difficulties.append({
        'target_word': tword,
        'taboos':      ', '.join(taboo_words),
        'difficulty':  round(np.mean(sims), 3),
    })

diff_df = pd.DataFrame(difficulties).sort_values('difficulty', ascending=False)

med = diff_df['difficulty'].median()
diff_df['tier'] = diff_df['difficulty'].apply(
    lambda d: 'hard' if d > diff_df['difficulty'].quantile(0.75)
              else 'medium' if d > med else 'easy'
)
TIER_COLOR = {'hard': '#da1e28', 'medium': '#f1c21b', 'easy': '#42be65'}

fig = px.bar(
    diff_df, x='difficulty', y='target_word', color='tier',
    color_discrete_map=TIER_COLOR,
    orientation='h',
    hover_data={'target_word': True, 'taboos': True, 'difficulty': ':.3f', 'tier': False},
    title='Taboo constraint difficulty per target — hover to see taboo words',
    labels={'difficulty': 'Mean similarity (target ↔ taboos)', 'target_word': ''},
)
fig.add_vline(x=med, line_dash='dash', line_color='#8d8d8d',
              annotation_text='median', annotation_font_color='#8d8d8d')
fig.update_layout(**PLOTLY_DARK, height=620, showlegend=True,
                  legend_title_text='Difficulty tier')
fig.show()

print('\nTop 5 hardest:')
print(diff_df.head(5)[['target_word', 'taboos', 'difficulty']].to_string(index=False))
print('\nTop 5 easiest:')
print(diff_df.tail(5)[['target_word', 'taboos', 'difficulty']].to_string(index=False))

## 7 — Does difficulty predict trial duration?

If the semantic constraint analysis captures real difficulty, harder targets should  
take longer in the actual experiment.  Let's check.

In [ ]:
if ('trial_end_time' in meta.columns
        and 'trial_number' in meta.columns
        and 'pair_id' in meta.columns):

    meta_sorted = meta.sort_values(['pair_id', 'trial_number'])
    meta_sorted = meta_sorted.copy()
    meta_sorted['end_dt'] = pd.to_datetime(
        meta_sorted['trial_end_time'], dayfirst=True, errors='coerce'
    )
    meta_sorted['duration_s'] = (
        meta_sorted.groupby('pair_id')['end_dt']
        .diff().dt.total_seconds()
    )
    durations = (
        meta_sorted.groupby('target_word')['duration_s']
        .median().reset_index()
        .rename(columns={'duration_s': 'median_duration_s'})
    )

    # diff_df uses 'target_word'; join on that key
    merged = diff_df.merge(durations, on='target_word', how='left').dropna(
        subset=['median_duration_s']
    )

    if merged.empty:
        print('Not enough duration data after merging — '
              'trial_end_time may only have minute resolution.')
    else:
        corr = merged['difficulty'].corr(merged['median_duration_s'])
        fig = px.scatter(
            merged, x='difficulty', y='median_duration_s',
            text='target_word',
            hover_data={'target_word': True, 'taboos': True,
                        'difficulty': ':.3f', 'median_duration_s': ':.1f'},
            title=f'Semantic difficulty vs. trial duration  (r = {corr:.2f})',
            labels={
                'difficulty': 'Semantic constraint difficulty',
                'median_duration_s': 'Median trial duration (s)',
            },
        )
        fig.update_traces(
            textposition='top center', textfont_size=8,
            marker=dict(color='#78a9ff', size=9),
        )
        fig.update_layout(**PLOTLY_DARK, height=480)
        fig.show()
else:
    print('Trial duration not available in the offline fallback — '
          'run with the full metadata.csv to see this plot.')

---
## What did we just do?

Without ever calling it "NLP", this notebook used:

| Concept | Where it appeared |
|---------|------------------|
| Word embeddings | Every `nlp(word).vector` call |
| Cosine similarity | `spacy_token.similarity()` |
| Dimensionality reduction | PCA to project 300D → 2D for plotting |
| Corpus analysis | Aggregating over the metadata CSV |
| Semantic distance as a measurement | The difficulty score |

---
## Go further

- **Try a different target** — change `TARGET` in cell 3 to any word in the corpus
- **Expand the candidate pool** — add more words to `CANDIDATE_POOL` and see what the computer would suggest
- **Cluster the targets** — use `sklearn.cluster.KMeans` on the target vectors. Do the clusters make sense?
- **Add the board/ground split** — does difficulty interact with physical condition in trial duration?
- **Try a larger model** — replace `en_core_web_md` with `en_core_web_lg` (685k vocab) or a sentence-transformer for better vectors